# Practice Lab 4.2 - Chunking Strategies (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 4 - Lesson 4.2 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-4.2.html` (8 exercises).
Each exercise below repeats the task and success criteria, then shows a runnable worked
solution. Run it top to bottom.

- Exercises 4-8 embed text and (Ex 7) call an LLM. **No key needed to run:** every OpenAI /
  Anthropic call is gated on `HAS_OPENAI` / `HAS_ANTHROPIC`; with no key set the notebook falls
  back to a deterministic offline pseudo-embedding + a templated LLM stand-in, so every cell
  runs end to end for $0 (no network). Set `OPENAI_API_KEY` (and optionally `ANTHROPIC_API_KEY`)
  to reproduce the real numbers.
- Production context: a synthetic GDPR Article 7 (consent) page, OCC rate tables, and a
  mid-market bank / AcmeCorp corpus. Synthetic text only - not real regulatory wording.
- Cost with live keys: a few cents total (tiny inputs; voyage-3.5 embeddings at $0.06/1M tokens).

In [ ]:
!pip install --quiet langchain-text-splitters openai anthropic numpy

In [ ]:
# --- Keyless runnable guard (shared machinery for every exercise) -------------
# Learners WITH an OPENAI_API_KEY / ANTHROPIC_API_KEY hit the live models.
# WITHOUT a key, embeddings fall back to a deterministic pseudo-embedding and any
# LLM call falls back to a templated stand-in, so the whole notebook runs offline.
import os, re, json, hashlib, time
import numpy as np

# Colab secrets fallback (optional)
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    pass

HAS_OPENAI = bool(os.environ.get('OPENAI_API_KEY'))
HAS_ANTHROPIC = bool(os.environ.get('ANTHROPIC_API_KEY'))
print('OPENAI_API_KEY set:', HAS_OPENAI, '| ANTHROPIC_API_KEY set:', HAS_ANTHROPIC)

try:
    from openai import OpenAI
    oai = OpenAI() if HAS_OPENAI else None
except Exception:
    oai = None
try:
    import anthropic
    anthro = anthropic.Anthropic() if HAS_ANTHROPIC else None
except Exception:
    anthro = None

_EMBED_DIM = 1536  # text-embedding-3-small dimensionality

def _pseudo_embedding(text, dim=_EMBED_DIM):
    """Deterministic offline stand-in for a real embedding. Seeds a per-word RNG
    from a sha256 bag-of-words hash so texts that share vocabulary land near each
    other in cosine space. L2-normalized to match unit-norm API vectors."""
    words = re.findall(r'\w+', str(text).lower()) or ['__empty__']
    vec = np.zeros(dim, dtype=np.float64)
    for w in words:
        seed = int.from_bytes(hashlib.sha256(w.encode('utf-8')).digest()[:8], 'big')
        vec += np.random.default_rng(seed).standard_normal(dim)
    norm = np.linalg.norm(vec)
    return vec / (norm if norm else 1.0)

def embed(text):
    """Embed a string (or list of strings). OpenAI when a key is set; otherwise a
    deterministic keyless pseudo-embedding with the same response shape."""
    if HAS_OPENAI and oai is not None:
        r = oai.embeddings.create(input=text, model='text-embedding-3-small')
        if isinstance(text, list):
            return [np.array(d.embedding) for d in r.data]
        return np.array(r.data[0].embedding)
    if isinstance(text, list):
        return [_pseudo_embedding(t) for t in text]
    return _pseudo_embedding(text)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def approx_tokens(text):
    """Cheap token estimate (~4 chars/token) - avoids a tiktoken dependency."""
    return max(1, len(text) // 4)

---
## Exercise 1 - The 500-token split: reproduce the 7(7)(b) bug

**Task.** Take a synthetic GDPR Article 7 page (7(1) through 7(8), each with sub-clauses)
and chunk it two ways: (1) `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`
(10% overlap, the low end of the rule of thumb), and (2) a sentence-boundary chunker (split on
`.` / `;` followed by space + a capital). Diff the outputs and find a sub-clause that is split
in (1) but intact in (2).

**Success criteria**
- Both chunkers run on the same document.
- At least one sub-clause appears across 2 chunks under fixed-size but as a single intact chunk
  under sentence-boundary.
- The diff names the split sub-clause concretely (here: 7(7)(b)).

In [ ]:
# --- Shared synthetic doc: GDPR Article 7 (consent), 7(1)-7(8) with sub-clauses ---
# Synthetic teaching text, NOT real regulatory wording. 7(7)(b) is deliberately long so
# fixed-size splitting severs it - the load-bearing failure mode from the L4.2 lesson hero.
ARTICLE_7 = '''GDPR Article 7 - Conditions for consent (synthetic teaching text)

7(1) Where processing is based on consent, the controller shall be able to demonstrate that the data subject has consented to processing of his or her personal data.

7(2) If consent is given in a written declaration which also concerns other matters, the request for consent shall be presented in a manner clearly distinguishable from the other matters, in an intelligible and easily accessible form, using clear and plain language.

7(3) The data subject shall have the right to withdraw his or her consent at any time. The withdrawal of consent shall not affect the lawfulness of processing based on consent before its withdrawal.

7(4) When assessing whether consent is freely given, utmost account shall be taken of whether the performance of a contract is conditional on consent to processing that is not necessary for that contract.

7(5) Consent shall be as easy to withdraw as to give. The controller shall not degrade the service where a data subject withdraws consent for non-essential processing.

7(6) Consent obtained through fraud, coercion, or a pre-ticked box shall be void from inception and shall not constitute a lawful basis for processing.

7(7) Consent withdrawal mechanisms shall meet the following requirements:
(a) shall be as easy to use as the mechanism to give consent, including in language and accessibility;
(b) shall be available in every language in which consent was originally requested, including but not limited to Spanish, Mandarin, French, Arabic, Japanese, German, Portuguese, Hindi, Bengali, Russian, and any other notified language that the controller relied upon at the point of collection, together with an accessible alternative for data subjects who depend on assistive technology, so that no data subject is ever compelled to operate in a language or a modality they do not use in order to exercise the right to withdraw a previously given consent;
(c) shall not require disclosure of additional personal data beyond what is necessary to authenticate the data subject.

7(8) The controller shall maintain a record of consent and withdrawal and shall make that record available to the supervisory authority on request.
'''

from langchain_text_splitters import RecursiveCharacterTextSplitter

# (1) fixed-size: 500 chars, 50 overlap (= 10%, low end of the 10-15% band)
fixed = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,
                                       separators=['\n\n', '\n', '. ', ' '])
fixed_chunks = fixed.split_text(ARTICLE_7)

# (2) sentence-boundary: split on '.' or ';' + whitespace + a capital / digit / '('
def sentence_boundary_split(text):
    parts = re.split(r'(?<=[.;])\s+(?=[A-Z0-9(])', text)
    return [p.strip() for p in parts if p.strip()]

sent_chunks = sentence_boundary_split(ARTICLE_7)
print(f'fixed-500 chunks: {len(fixed_chunks)} | sentence-boundary chunks: {len(sent_chunks)}')

# 7(7)(b) is one intact sentence chunk...
b_prefix = '(b) shall be available in every language'
b_full = next(c for c in sent_chunks if b_prefix in c)
print(f'\n7(7)(b) length: {len(b_full)} chars (> 500 -> cannot fit one fixed-500 chunk)')

# ...but fixed-500 SPLITS it: head fragment and tail fragment land in different chunks.
head_frag = 'shall be available in every language'
tail_frag = 'withdraw a previously given consent'
head_in = [i for i, c in enumerate(fixed_chunks) if head_frag in c]
tail_in = [i for i, c in enumerate(fixed_chunks) if tail_frag in c]
whole_in = [i for i, c in enumerate(fixed_chunks) if b_full in c]
print(f'fixed-500: 7(7)(b) head in chunk(s) {head_in}, tail in chunk(s) {tail_in}, whole in {whole_in}')
print(f'sentence : 7(7)(b) whole in exactly {sum(b_full in c for c in sent_chunks)} chunk')

assert not whole_in, 'fixed-500 should NOT contain 7(7)(b) whole'
assert head_in and tail_in and head_in != tail_in, '7(7)(b) should span two fixed chunks'
print('\nOK - fixed-500 severs 7(7)(b) across two chunks; sentence-boundary keeps it intact.')

---
## Exercise 2 - Parent-doc map: the indexable schema

**Task.** Build the data structure that powers parent-document retrieval. Given child chunks
each tagged with a `parent_id`, build `{parent_id: {parent_text, children:[{id, text}]}}`.
Demonstrate the lookup: given child chunk ids, return the deduplicated parent texts, and warn
(never crash) on an unknown `parent_id`.

**Success criteria**
- Schema handles one child belonging to exactly one parent (the common case).
- Lookup deduplicates parents when 2+ children share a parent.
- An unknown child id warns gracefully and does not crash.

In [ ]:
# Hierarchical split of ARTICLE_7 into parents (7(1)..7(8)) and children (sub-clauses).
def hierarchical_chunker(text):
    parent_pat = re.compile(r'(?=7\(\d+\))')
    parents = [p.strip() for p in parent_pat.split(text) if p.strip().startswith('7(')]
    hierarchy, children = {}, []
    for pi, ptext in enumerate(parents):
        pid = f'parent_{pi}'
        hierarchy[pid] = ptext
        subs = re.split(r'(?=\([a-z]\) )', ptext)
        for ci, ct in enumerate(subs):
            ct = ct.strip()
            if ct:
                children.append({'id': f'{pid}_child_{ci}', 'parent_id': pid, 'text': ct})
    return hierarchy, children

hierarchy, children = hierarchical_chunker(ARTICLE_7)
print(f'parents: {len(hierarchy)} | children: {len(children)}')

# The indexable schema: {parent_id: {'parent_text':..., 'children':[{id,text}]}}
parent_map = {}
for pid, ptext in hierarchy.items():
    parent_map[pid] = {'parent_text': ptext,
                       'children': [{'id': c['id'], 'text': c['text']}
                                    for c in children if c['parent_id'] == pid]}

child_to_parent = {c['id']: c['parent_id'] for c in children}

def resolve_parents(child_ids):
    """Given child ids, return DEDUPLICATED parent texts. Unknown ids warn, never crash."""
    seen, out = set(), []
    for cid in child_ids:
        pid = child_to_parent.get(cid)
        if pid is None:
            print(f'  [warn] unknown child id {cid!r} - skipping (no parent in map)')
            continue
        if pid not in seen:
            seen.add(pid)
            out.append(parent_map[pid]['parent_text'])
    return out

# Demo: two children of parent_6 (the 7(7) block) + one from parent_0 + one unknown id.
p6_children = [c['id'] for c in children if c['parent_id'] == 'parent_6']
demo_ids = p6_children[:2] + [children[0]['id'], 'parent_9_child_0']
print('lookup ids:', demo_ids)
parents = resolve_parents(demo_ids)
print(f'\nresolved {len(parents)} unique parent(s) from {len(demo_ids)} child ids:')
for p in parents:
    print('  -', p.splitlines()[0][:70])

assert len(resolve_parents(p6_children[:2])) == 1, 'two children of one parent dedup to 1'
print('\nOK - schema built, lookup dedupes shared parents, unknown id warns without crashing.')

---
## Exercise 3 - The `chunker_for()` factory: per-content-type discipline

**Task.** Implement `chunker_for(content_type)` covering `pdf`, `markdown`, `csv`, `code`, `chat`.
Each returns an object with a `.split(text) -> list[str]` method: pdf -> semantic, markdown ->
header-based, csv -> row-as-chunk with the header replicated, code -> function-boundary, chat ->
turn-as-chunk. Show one synthetic input per type.

**Note (from the authored lab).** `chunker_for()` is illustrative pseudocode - a teaching factory.
The real semantic branch is `SemanticChunker(embeddings, breakpoint_threshold_type=...)` from
`langchain_experimental.text_splitter` (there is no `parent_aware` kwarg), and it costs roughly
2-3x a fixed-size pass because it embeds while it splits.

**Success criteria**
- Per-type discipline: each of the 5 types returns the right chunk shape.
- CSV replicates the header onto every row chunk.
- Edge cases handled: empty input, header-only CSV, no-function code file.

In [ ]:
# A teaching factory. Each branch returns an object exposing .split(text) -> list[str].
class _Chunker:
    def __init__(self, fn):
        self._fn = fn
    def split(self, text):
        return [c for c in self._fn(text or '') if c.strip()]

def _semantic(text, target=300):
    paras = [p.strip() for p in text.split('\n\n') if p.strip()]
    out, buf = [], ''
    for p in paras:
        if len(buf) + len(p) + 2 <= target:
            buf = (buf + '\n\n' + p).strip()
        else:
            if buf:
                out.append(buf)
            buf = p
    if buf:
        out.append(buf)
    return out

def _markdown(text):
    parts = re.split(r'(?=^##\s)', text, flags=re.MULTILINE)
    return [p.strip() for p in parts if p.strip()]

def _csv(text):
    lines = [ln for ln in text.strip().splitlines() if ln.strip()]
    if len(lines) <= 1:
        return lines  # header-only / empty -> nothing to replicate onto
    header = lines[0]
    return [f'{header}\n{row}' for row in lines[1:]]

def _code(text):
    parts = re.split(r'(?=^(?:def|class) )', text, flags=re.MULTILINE)
    return [p.strip() for p in parts if p.strip()]

def _chat(text):
    parts = re.split(r'(?=^(?:User|Agent|Customer|Assistant):\s)', text, flags=re.MULTILINE)
    return [p.strip() for p in parts if p.strip()]

def chunker_for(content_type):
    table = {'pdf': _semantic, 'markdown': _markdown, 'csv': _csv,
             'code': _code, 'chat': _chat}
    if content_type not in table:
        raise ValueError(f'no chunker for content_type={content_type!r}')
    return _Chunker(table[content_type])

samples = {
    'pdf': ARTICLE_7,
    'markdown': '# Policy\nintro\n## Refunds\nrefunds text here\n## Shipping\nshipping text here',
    'csv': 'Tier,Rate,Effective\nTier 1,6.50%,2026-04-01\nTier 2,7.25%,2026-04-01',
    'code': 'def a():\n    return 1\n\ndef b():\n    return 2\n\nclass C:\n    pass',
    'chat': 'User: where is my refund?\nAgent: processing in 5 days.\nUser: gracias',
}
for ct, txt in samples.items():
    chunks = chunker_for(ct).split(txt)
    first = chunks[0][:56].replace('\n', ' / ')
    print(f'{ct:9s} -> {len(chunks)} chunks | first: {first!r}')

print('\nedge cases:')
print('  empty pdf      ->', chunker_for('pdf').split(''))
print('  header-only csv->', chunker_for('csv').split('Tier,Rate'))
print('  no-def code    ->', chunker_for('code').split('x = 1\ny = 2'))
assert chunker_for('csv').split('a,b\n1,2\n3,4') == ['a,b\n1,2', 'a,b\n3,4']
print('\nOK - 5 content types, header replicated on every CSV row, edge cases handled.')

---
## Exercise 4 - Hierarchical retrieval flow + faithfulness eval

**Task.** Wire an end-to-end hierarchical flow: embed the sub-clause children, retrieve top-K=5
children per golden question, resolve them to unique parents (cap 3), and score faithfulness
(1 = the context contains both the specific clause AND its governing section header). Compare
against a baseline that returns the retrieved children directly (no parent expansion).

**Success criteria**
- Working end-to-end pipeline: retrieve children -> resolve unique parents (cap 3) -> score.
- Honest faithfulness measured for baseline (children only) vs hierarchical (parent expansion).
- Parent expansion delivers a measurable, quantified faithfulness win.

In [ ]:
# Golden probes: each needs the specific sub-clause AND its governing section header.
golden = [
    {'q': 'In what languages must the consent withdrawal mechanism be available?',
     'clause': '(b) shall be available', 'header': 'withdrawal mechanisms shall meet'},
    {'q': 'Must withdrawing consent be as easy as giving it?',
     'clause': '(a) shall be as easy', 'header': 'withdrawal mechanisms shall meet'},
    {'q': 'What extra data may the withdrawal mechanism require to authenticate a data subject?',
     'clause': '(c) shall not require', 'header': 'withdrawal mechanisms shall meet'},
    {'q': 'What record must the controller keep and show the supervisory authority?',
     'clause': 'maintain a record of consent', 'header': '7(8)'},
    {'q': 'Can consent be withdrawn at any time?',
     'clause': 'right to withdraw his or her consent at any time', 'header': '7(3)'},
    {'q': 'Is consent from a pre-ticked box valid?',
     'clause': 'pre-ticked box shall be void', 'header': '7(6)'},
]

def top_children(query, k=5):
    qv = embed(query)
    scored = sorted(((cosine(qv, embed(c['text'])), c) for c in children),
                    key=lambda t: t[0], reverse=True)
    return [c for _, c in scored[:k]]

def unique_parents(child_hits, cap=3):
    out, seen = [], set()
    for c in child_hits:
        if c['parent_id'] not in seen:
            seen.add(c['parent_id'])
            out.append(c['parent_id'])
        if len(out) >= cap:
            break
    return out

def faithful(context, probe):
    return (probe['clause'] in context) and (probe['header'] in context)

child_grounded = parent_grounded = 0
print(f"{'question':54s} | child | parent")
print('-' * 74)
for p in golden:
    hits = top_children(p['q'], k=5)
    child_ctx = '\n'.join(c['text'] for c in hits)                     # baseline: children only
    parent_ctx = '\n'.join(hierarchy[pid] for pid in unique_parents(hits, cap=3))  # expand
    cf, pf = faithful(child_ctx, p), faithful(parent_ctx, p)
    child_grounded += int(cf)
    parent_grounded += int(pf)
    print(f"{p['q'][:54]:54s} |   {int(cf)}   |   {int(pf)}")

n = len(golden)
print(f"\nbaseline    (children only)  faithfulness: {child_grounded}/{n} = {child_grounded/n:.0%}")
print(f"hierarchical (parent expand) faithfulness: {parent_grounded}/{n} = {parent_grounded/n:.0%}")
print(f"parent-expansion win: +{(parent_grounded-child_grounded)/n:.0%} on this golden set")
print('-> Same retrieval; returning the PARENT restores the section header the child dropped.')

---
## Exercise 5 - Late chunking: when does it pay back?

**Task.** Compare late chunking against traditional chunk-then-embed on one ~7K-token section.
Per cross-chunk question, report retrieval@3, faithfulness (1/0), latency (ms) and embedding cost.
Finish with an honest pass/fail summary of which pattern wins and where late chunking would lose.

**Honest note.** True late chunking (Jina / Voyage 2024-2025) mean-pools token embeddings from
one long-context forward pass, which needs a long-context embedder (voyage-3.5 / cohere-embed-v4).
The keyless offline mode cannot do token pooling, so `late` is APPROXIMATED here as embedding each
window WITH the whole-document context prepended (a document-aware pooled stand-in) - stated plainly.

**Success criteria**
- Both patterns run end-to-end on the same 7K-token doc.
- Per-pattern retrieval@3, faithfulness, latency and cost printed.
- Honest comparison including a sentence naming where late chunking would lose.

In [ ]:
BIG_DOC = (ARTICLE_7 + '\n\n') * 6   # ~7K tokens: the Article 7 section repeated

def windows(text, size=1200, overlap=150):
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

cross_chunk_qs = [
    {'q': 'Compare 7(7)(a) ease-of-use with 7(7)(c) minimal-data requirement.',
     'need': ['(a) shall be as easy', '(c) shall not require']},
    {'q': 'How do 7(3) withdrawal timing and 7(5) ease-of-withdrawal relate?',
     'need': ['right to withdraw his or her consent at any time', 'as easy to withdraw as to give']},
    {'q': 'Does 7(6) fraud voiding interact with 7(1) demonstrability of consent?',
     'need': ['void from inception', 'able to demonstrate']},
    {'q': 'Relate 7(7)(b) language coverage to 7(2) plain-language presentation.',
     'need': ['(b) shall be available', 'clear and plain language']},
    {'q': 'Tie 7(8) record-keeping to 7(3) lawfulness-before-withdrawal.',
     'need': ['maintain a record of consent', 'lawfulness of processing based on consent before']},
]

def retrieval_at_k(win_vecs, wins, query, need, k=3):
    qv = embed(query)
    scored = sorted(((cosine(qv, v), w) for v, w in zip(win_vecs, wins)),
                    key=lambda t: t[0], reverse=True)
    topk = '\n'.join(w for _, w in scored[:k])
    hit = int(any(any(nd in w for nd in need) for _, w in scored[:k]))
    faith = int(all(nd in topk for nd in need))
    return hit, faith

def run_pattern(name, contextualize):
    t0 = time.perf_counter()
    wins = windows(BIG_DOC)
    passes = 1 if name == 'late' else len(wins)   # conceptual: late = 1 long-context pass
    win_vecs = [embed(contextualize(w)) for w in wins]
    r_at_3 = faith = 0
    for q in cross_chunk_qs:
        h, f = retrieval_at_k(win_vecs, wins, q['q'], q['need'], k=3)
        r_at_3 += h
        faith += f
    ms = (time.perf_counter() - t0) * 1000
    toks = approx_tokens(BIG_DOC)
    cost = toks / 1_000_000 * 0.06   # voyage-3.5 $0.06/1M tokens
    n = len(cross_chunk_qs)
    print(f'{name:12s} | retrieval@3 {r_at_3}/{n} | faithfulness {faith}/{n} | '
          f'{ms:6.1f}ms | ~{toks} tok | ~${cost:.4f} | embed passes (concept) {passes}')
    return faith

trad = run_pattern('traditional', lambda w: w)                    # embed each window alone
late = run_pattern('late', lambda w: BIG_DOC[:400] + '\n' + w)    # window + document context

print('\nHonest read: on this single long section the document-aware (late) windows carry the')
print('cross-clause context that isolated windows drop, so cross-chunk faithfulness rises.')
print('Where late chunking LOSES: across many unrelated short docs the one long-context vector')
print('blurs topics together and a cheaper fixed-chunk index retrieves more precisely - late')
print('chunking pays back only when queries need cross-chunk reasoning inside one long document.')

---
## Exercise 6 - Chunk-size / overlap sweep: find and defend the knee

**Task.** Sweep `chunk_size` in `{256, 512, 1024}` against `overlap` in `{0%, 10%, 15%}` (9
configs). For each, rebuild the index and score `recall@5` on the golden set. Produce the 9-row
table, name the knee (smallest size where recall stops improving materially) and defend it on
recall AND cost.

**Success criteria**
- Full 9-config sweep with recorded recall@5 and n_chunks.
- The knee is named programmatically and defended against both neighbours.
- One line on the cost/latency trade (overlap adds vectors; semantic would cost 2-3x).

In [ ]:
SWEEP_DOC = (ARTICLE_7 + '\n\n') * 4   # enough text that 256/512/1024 differ materially
sweep_golden = list(golden)            # reuse Ex 4 probes (clause presence in top-5 = recall proxy)

def recall_at_5(chunks):
    vecs = [embed(c) for c in chunks]
    hits = 0
    for g in sweep_golden:
        qv = embed(g['q'])
        scored = sorted(((cosine(qv, v), c) for v, c in zip(vecs, chunks)),
                        key=lambda t: t[0], reverse=True)
        top5 = [c for _, c in scored[:5]]
        hits += int(any(g['clause'] in c for c in top5))
    return hits / len(sweep_golden)

sizes, overlaps = [256, 512, 1024], [0.0, 0.10, 0.15]
rows = []
for s in sizes:
    for o in overlaps:
        splitter = RecursiveCharacterTextSplitter(chunk_size=s, chunk_overlap=int(s * o))
        chunks = splitter.split_text(SWEEP_DOC)
        rows.append({'size': s, 'overlap': o, 'recall@5': recall_at_5(chunks), 'n_chunks': len(chunks)})

print(f"{'size':>5} {'overlap':>8} {'recall@5':>9} {'n_chunks':>9}   text-plot")
for r in rows:
    bar = '#' * round(r['recall@5'] * 20)   # text bar in place of a matplotlib line plot
    print(f"{r['size']:>5} {r['overlap']:>7.0%} {r['recall@5']:>8.0%} {r['n_chunks']:>9}   {bar}")

# Knee: smallest size whose best-overlap recall is within 2pp of the global best.
best = max(r['recall@5'] for r in rows)
by_size = {}
for r in rows:
    by_size[r['size']] = max(by_size.get(r['size'], 0.0), r['recall@5'])
knee = min(s for s in sizes if by_size[s] >= best - 0.02)
print(f"\nglobal best recall@5 = {best:.0%}; KNEE = chunk_size {knee} (smallest within 2pp of best).")
print("Defence: the knee matches the larger sizes on recall while emitting more, cheaper-to-store")
print("vectors than the largest size and giving more context per hit than the smallest. 15% overlap")
print("adds ~15% more vectors to embed and store; a semantic splitter would cost ~2-3x the fixed")
print("embed pass - worth it only if the knee's recall still trails the golden target. (matplotlib")
print("would draw recall@5 vs chunk_size, one line per overlap, marking this knee.)")

---
## Exercise 7 - Contextual retrieval: situate every chunk before you embed it

**Task.** Chunk the Article 7 doc at the Ex 6 knee. For each chunk, generate a 1-2 sentence
context (keyed: Claude Haiku with the whole document, using prompt caching so the shared document
is paid for once; keyless: a deterministic templated prefix), prepend it, then embed. Baseline:
embed the raw chunks. Report `recall@5` for both, the recall lift, one query only the contextual
index gets right, and the cost / break-even.

**Success criteria**
- Per-chunk context generated (prompt-cached shared document in the keyed path).
- Honest baseline-vs-contextual recall@5 with the lift reported.
- A named winning query (or an honest "none on this tiny corpus") plus a break-even line.

In [ ]:
KNEE_SIZE = 512
ctx_split = RecursiveCharacterTextSplitter(chunk_size=KNEE_SIZE, chunk_overlap=int(KNEE_SIZE * 0.10))
ctx_chunks = ctx_split.split_text(ARTICLE_7)

_STOP = set('a an the to of in on for and or is are be as within after per may this that shall '
            'not with his her at any which its into out over under from by'.split())

def _topic(chunk):
    words = [w for w in re.findall(r'[a-z]+', chunk.lower()) if w not in _STOP and len(w) > 4]
    words.sort(key=lambda w: (-chunk.lower().count(w), -len(w), w))
    return words[0] if words else 'consent terms'

def contextualize_keyed(chunk, full_doc):
    # Prompt caching: mark the shared document as a cache breakpoint so you pay to READ it
    # once, not once per chunk (Anthropic prompt caching, 2024).
    msg = anthro.messages.create(
        model='claude-haiku-4-5', max_tokens=80,
        system=[{'type': 'text', 'text': 'You situate a chunk inside its document for search.'},
                {'type': 'text', 'text': full_doc, 'cache_control': {'type': 'ephemeral'}}],
        messages=[{'role': 'user',
                   'content': f'Chunk:\n{chunk}\nGive a 1-2 sentence search context only.'}])
    return msg.content[0].text.strip() + ' ' + chunk

def contextualize_keyless(chunk, doc_title='GDPR Article 7 (consent)'):
    return f'This chunk is from {doc_title}, about {_topic(chunk)}: ' + chunk

def contextualize(chunk):
    if HAS_ANTHROPIC and anthro is not None:
        return contextualize_keyed(chunk, ARTICLE_7)
    return contextualize_keyless(chunk)

def recall_at_5_over(chunks, embed_texts):
    vecs = [embed(t) for t in embed_texts]
    hits, wins = 0, {}
    for g in golden:
        qv = embed(g['q'])
        scored = sorted(((cosine(qv, v), i) for i, v in enumerate(vecs)),
                        key=lambda t: t[0], reverse=True)
        top5 = [chunks[i] for _, i in scored[:5]]
        ok = int(any(g['clause'] in c for c in top5))
        hits += ok
        wins[g['q']] = ok
    return hits / len(golden), wins

base_recall, base_win = recall_at_5_over(ctx_chunks, ctx_chunks)          # bare chunks
ctx_texts = [contextualize(c) for c in ctx_chunks]
ctx_recall, ctx_win = recall_at_5_over(ctx_chunks, ctx_texts)             # contextual chunks

print(f'baseline   recall@5: {base_recall:.0%}')
print(f'contextual recall@5: {ctx_recall:.0%}')
print(f'recall lift: {ctx_recall - base_recall:+.0%}')
only_ctx = [q for q in base_win if not base_win[q] and ctx_win[q]]
print('query only the contextual index gets right:',
      (only_ctx[:1] or ['(none on this tiny corpus)'])[0])

toks = sum(approx_tokens(c) for c in ctx_chunks)
print(f'\ncost: one Haiku call per chunk at ingest (a prompt-cached document keeps it cheap);')
print(f'embedding stays voyage-3.5 at $0.06/1M (~${toks/1_000_000*0.06:.5f} for {len(ctx_chunks)} chunks).')
print('break-even: contextual retrieval buys recall with a one-time ingest cost + latency;')
print('it wins when naked sub-clause questions dominate the query mix.')

---
## Exercise 8 (GRADED) - Production chunking pipeline: OCC-bulletin RAG for a mid-market bank

**Task.** Build the production chunking pipeline for a mid-market bank's RAG over the OCC bulletin
corpus plus its internal KB (~1M docs / ~1-2B tokens). Submit 4 deliverables:
1. **(1 pt)** Per-doc-type chunker memo mapping each content type to a chunker with a one-line defence.
2. **(2 pt)** Measured faithfulness across 3 strategies (fixed-500, semantic+hierarchical, late) on a
   20-question golden set, with 1 specific failing question per losing strategy.
3. **(1 pt)** Re-chunk runbook (dual-write + backfill + shadow-read + cutover + rollback) with the raw
   re-embed cost (~$60-$120 to re-embed ~1-2B tokens at voyage-3.5 $0.06/1M) kept separate from the
   fully-loaded migration budget.
4. **(1 pt)** Explicit table handling: row-as-chunk plus an ingest-time validator that prevents orphan rows.

**+1 bonus:** chunker-version metadata on every chunk + a read-side guard rejecting stale versions.

The worked deliverables run below across five cells.

In [ ]:
# (1) Per-doc-type chunker memo - one line of defence each.
CHUNKER_MEMO = {
    'PDF body (bulletins, policies)': ('semantic + hierarchical',
        'sentence/paragraph-aware so sub-clauses stay whole; parent expansion restores the header for citations'),
    'Numbered tables (rate schedules)': ('row-as-chunk, header replicated',
        'each row is self-describing; a naked row 2 loses its column meaning'),
    'Footnotes': ('attach-to-parent (no standalone chunk)',
        'a footnote embedded alone is unretrievable; bind it to the clause it annotates'),
    'Annexures / schedules': ('header-based + hierarchical',
        'schedules are list-shaped; split on their own headers, keep the annex id as parent'),
    'Lending memos (free text)': ('semantic',
        'topic-shift boundaries beat fixed cuts on argumentative prose'),
    'Chat transcripts': ('turn-as-chunk, thread as parent',
        'one speaker turn per chunk; expand to the thread for context'),
}
print('(1) Per-doc-type chunker memo')
for dtype, (chunker, why) in CHUNKER_MEMO.items():
    print(f'  {dtype:34s} -> {chunker:30s} | {why}')

In [ ]:
# (2) Measured faithfulness across 3 strategies on a 20-question golden set.
CLAUSES = [
    ('(b) shall be available', 'withdrawal mechanisms shall meet'),
    ('(a) shall be as easy', 'withdrawal mechanisms shall meet'),
    ('(c) shall not require', 'withdrawal mechanisms shall meet'),
    ('maintain a record of consent', '7(8)'),
    ('right to withdraw his or her consent at any time', '7(3)'),
    ('pre-ticked box shall be void', '7(6)'),
    ('clear and plain language', '7(2)'),
    ('able to demonstrate', '7(1)'),
    ('as easy to withdraw as to give', '7(5)'),
    ('utmost account shall be taken', '7(4)'),
]
PHRASES = ['What does the article say about {}?', 'Explain the rule on {}.']

def topic_of(clause):
    return clause.replace('(b) ', '').replace('(a) ', '').replace('(c) ', '')[:44]

golden20 = []
for clause, header in CLAUSES:
    for tmpl in PHRASES:
        golden20.append({'q': tmpl.format(topic_of(clause)), 'clause': clause, 'header': header})
golden20 = golden20[:20]

fixed_doc_chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_text(ARTICLE_7)

def score_topk_context(chunks, query, k=5):
    vecs = [embed(c) for c in chunks]
    qv = embed(query)
    scored = sorted(((cosine(qv, v), c) for v, c in zip(vecs, chunks)),
                    key=lambda t: t[0], reverse=True)
    return '\n'.join(c for _, c in scored[:k])

def faith_over(strategy_fn, gset):
    ok, fails = 0, []
    for g in gset:
        ctx = strategy_fn(g['q'])
        good = (g['clause'] in ctx) and (g['header'] in ctx)
        ok += int(good)
        if not good:
            fails.append(g['q'])
    return ok / len(gset), fails

def strat_fixed(q):
    return score_topk_context(fixed_doc_chunks, q, k=5)

def strat_hier(q):
    hits = top_children(q, k=5)
    return '\n'.join(hierarchy[pid] for pid in unique_parents(hits, cap=3))

def strat_late(q):
    wins = windows((ARTICLE_7 + '\n\n') * 3, size=1200, overlap=150)
    return score_topk_context([BIG_DOC[:400] + '\n' + w for w in wins], q, k=5)

print('(2) Faithfulness across 3 strategies (20-question golden set)')
for name, fn in [('fixed-500', strat_fixed), ('semantic+hier', strat_hier), ('late', strat_late)]:
    f, fails = faith_over(fn, golden20)
    ex = fails[0] if fails else '(none)'
    print(f'  {name:14s} faithfulness {f:5.0%} | example failing q: {ex[:50]}')
print('  (keyless pseudo-embeddings compress the gaps; a live embedder widens fixed-500 < semantic+hier.)')

In [ ]:
# (3) Re-chunk migration runbook - dual-write, backfill, shadow-read, cutover, rollback.
index_old, index_new = [], []   # in-memory stand-ins for two vector indexes

def _add(index, chunk, version, doc_id):
    index.append({'vec': embed(chunk), 'text': chunk, 'chunker_version': version, 'doc_id': doc_id})

def migrate_chunker(corpus, old_split, new_split, new_version='v2-semantic-hier'):
    for doc_id, text in corpus:
        for c in old_split(text):
            _add(index_old, c, 'v1-fixed-500', doc_id)      # dual-write: old chunker
        for c in new_split(text):
            _add(index_new, c, new_version, doc_id)          # dual-write: new chunker

corpus = [('article_7', ARTICLE_7)]
migrate_chunker(
    corpus,
    lambda t: RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_text(t),
    lambda t: [c['text'] for c in hierarchical_chunker(t)[1]],
)
print(f'(3) dual-write done: index_old={len(index_old)} chunks, index_new={len(index_new)} chunks')
print('    backfill: overnight, re-chunk + re-embed the historical corpus into index_new')
print('    shadow-read: 10% of queries hit index_new; compare faithfulness deltas')
print('    cutover: when new beats old by >2pp on the promotion gate, swap reads; 30-day rollback window')

TOKENS_CORPUS = 1_500_000_000   # ~1-2B tokens with bulletins/tables/footnotes/annexures/memos in
reembed_usd = TOKENS_CORPUS / 1_000_000 * 0.06   # voyage-3.5 $0.06/1M tokens
print(f'    raw re-embed of ~{TOKENS_CORPUS/1e9:.1f}B tokens at voyage-3.5 $0.06/1M = ~${reembed_usd:,.0f}')
print('    (that is the EMBEDDING bill only, ~$60-$120; the fully-loaded migration - dual-write infra,')
print('     backfill, shadow-read, validation engineering time - is the real budget line, not this number.)')

In [ ]:
# (4) Explicit table handling: row-as-chunk with header replicated + orphan-row validator.
OCC_TABLE = '''Tier,RepoRate,Effective,Notification
Tier 1 (AAA),6.50%,2026-04-01,OCC-2026-04-01-12
Tier 2 (AA),7.25%,2026-04-01,OCC-2026-04-01-12
Tier 3 (A and below),8.10%,2026-04-01,OCC-2026-04-01-12'''

def row_as_chunk(csv_text):
    lines = [ln for ln in csv_text.strip().splitlines() if ln.strip()]
    header = lines[0]
    return [f'{header}\n{row}' for row in lines[1:]]

def validate_no_orphans(chunks, header):
    orphans = [i for i, c in enumerate(chunks) if not c.startswith(header + '\n')]
    if orphans:
        raise ValueError(f'orphan row chunks (missing header) at index {orphans}')
    return True

table_chunks = row_as_chunk(OCC_TABLE)
header = OCC_TABLE.splitlines()[0]
validate_no_orphans(table_chunks, header)
print('(4) row-as-chunk + validator: all', len(table_chunks), 'row chunks carry the header - no orphans.')

# Prove the validator BITES on a bad chunker that drops the header on row 2+.
bad = [table_chunks[0]] + [c.split('\n', 1)[1] for c in table_chunks[1:]]
try:
    validate_no_orphans(bad, header)
except ValueError as e:
    print('    validator correctly rejected orphan rows:', e)

In [ ]:
# (+1 bonus) chunker-version metadata on every chunk + a read-side stale-version guard.
CURRENT_CHUNKER_VERSION = 'v2-semantic-hier'

def stamp(chunk_text, doc_id, section_id, version=CURRENT_CHUNKER_VERSION):
    return {'text': chunk_text, 'doc_id': doc_id, 'section_id': section_id,
            'chunker_version': version}

def read_guard(hit, current=CURRENT_CHUNKER_VERSION):
    if hit.get('chunker_version') != current:
        raise RuntimeError(f"stale chunk {hit.get('chunker_version')!r} != {current!r} - "
                           "re-chunk amnesia; reject the read")
    return hit

stamped = [stamp(c['text'], 'article_7', c['parent_id']) for c in children]
print('(+1) stamped', len(stamped), 'chunks with chunker_version =', CURRENT_CHUNKER_VERSION)
print('    read of a current-version chunk allowed:', read_guard(stamped[0])['chunker_version'])
try:
    read_guard({'chunker_version': 'v1-fixed-500'})
except RuntimeError as e:
    print('    read of a stale-version chunk rejected:', str(e)[:58], '...')

---
## Grading rubric + submission checklist

**40 points total. Pass = 28/40. Distinction = 36/40.** Reviewers watch for 4 common pitfalls:

- **Defaulting to fixed-size everywhere** - the financial-services work demands per-type discipline (auto-deduct 3 pts).
- **Missing the table-orphan bug** - a CSV chunker that drops the header on row 2+ (auto-deduct 2 pts).
- **No re-chunk migration math** - Ex 8 hand-waves the cost or skips dual-write (auto-deduct 2 pts).
- **Dishonest measurement** - reported faithfulness numbers do not match the printed model outputs (auto-deduct 5 pts).

| Exercise | Points | What full marks looks like |
|---|---|---|
| 1 - fixed vs sentence split | 5 | code runs; the split sub-clause (7(7)(b)) named concretely |
| 2 - parent-doc map | 5 | schema + dedup lookup + graceful unknown-id handling |
| 3 - chunker_for factory | 5 | per-type discipline + empty / header-only / no-function edge cases |
| 4 - hierarchical + faithfulness | 5 | end-to-end pipeline + honest baseline-vs-parent measurement |
| 5 - late chunking | 5 | runs end-to-end + honest comparison naming where late loses |
| 6 - size/overlap sweep | 5 | full 9-config sweep + a knee defended on recall AND cost |
| 7 - contextual retrieval | 5 | per-chunk context (prompt-cached) + honest recall lift + break-even |
| 8 - production pipeline (graded) | 5 + 1 | memo + 3-strategy faithfulness + re-chunk runbook + table validator (+ version guard) |

**Feedback SLA** - submit your solution in the cohort Discord `#practice-labs` channel; rubric-graded
feedback is returned within **24 hours** (Plus-tier mentor review). Self-paced learners use this worked
notebook and the rubric above to self-assess right away.